# 0 — Tokeniser Validation

Verify that all concept keywords tokenise as single tokens in each model's vocabulary.
If a keyword splits into multiple tokens, the ablation experiment (notebook 6) and
checker prompts (1B/R1B) may not work correctly for that concept.

**No GPU needed.** Loads tokenizers only.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=False)

from transformers import AutoTokenizer

MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}

PYTHON_KEYWORDS = {
    "import": "ast__Import", "from": "ast__ImportFrom",
    "break": "ast__Break", "pass": "ast__Pass", "continue": "ast__Continue",
    "assert": "ast__Assert", "if": "ast__If", "while": "ast__While",
    "for": "ast__For", "return": "ast__Return", "with": "ast__With",
    "raise": "ast__Raise", "del": "ast__Delete", "global": "ast__Global",
    "nonlocal": "ast__Nonlocal", "lambda": "ast__Lambda",
    "yield": "ast__Yield", "try": "ast__Try",
    "class": "ast__ClassDef", "def": "ast__FunctionDef",
    "async": "ast__AsyncFunctionDef",
    # Builtins
    "list": "builtin__list", "dict": "builtin__dict", "set": "builtin__set",
    "tuple": "builtin__tuple", "int": "builtin__int", "float": "builtin__float",
    "str": "builtin__str", "bool": "builtin__bool", "len": "builtin__len",
    "range": "builtin__range", "print": "builtin__print",
    "type": "builtin__type", "map": "builtin__map", "filter": "builtin__filter",
    "zip": "builtin__zip", "sorted": "builtin__sorted",
    "sum": "builtin__sum", "min": "builtin__min", "max": "builtin__max",
    "all": "builtin__all", "any": "builtin__any",
}

RUST_KEYWORDS = {
    "for": "rust__For", "while": "rust__While", "loop": "rust__Loop",
    "if": "rust__If", "match": "rust__Match",
    "fn": "rust__Fn", "let": "rust__Let",
    "struct": "rust__Struct", "enum": "rust__Enum",
    "impl": "rust__Impl", "trait": "rust__Trait",
    "use": "rust__Use", "mod": "rust__Mod",
    "return": "rust__Return", "break": "rust__Break", "continue": "rust__Continue",
    "const": "rust__Const", "static": "rust__Static",
    "async": "rust__Async", "await": "rust__Await",
    "unsafe": "rust__Unsafe",
    # Object keywords
    "Vec": "rust__Vec", "String": "rust__String",
    "Option": "rust__Option", "Result": "rust__Result",
    "HashMap": "rust__HashMap", "Box": "rust__Box",
    "Arc": "rust__Arc", "Rc": "rust__Rc",
    "Clone": "rust__Clone", "Iterator": "rust__Iterator",
    "Debug": "rust__Debug", "Display": "rust__Display",
    "Default": "rust__Default",
}

LANG_KEYWORDS = {"Python": PYTHON_KEYWORDS, "Rust": RUST_KEYWORDS}

print(f"Models: {list(MODEL_CONFIGS.keys())}")
print(f"Python keywords: {len(PYTHON_KEYWORDS)}")
print(f"Rust keywords: {len(RUST_KEYWORDS)}")

In [ ]:
# Cell 2 – Validate all keywords across all models

results = []

for model_key, config in MODEL_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Model: {config['id']} ({model_key})")
    print(f"{'='*60}")

    tok = AutoTokenizer.from_pretrained(config['id'], trust_remote_code=True)

    for lang_name, keywords in LANG_KEYWORDS.items():
        print(f"\n--- {lang_name} ---")
        n_ok, n_multi, n_total = 0, 0, len(keywords)

        for keyword, concept in sorted(keywords.items()):
            ids = tok.encode(keyword, add_special_tokens=False)
            decoded = [tok.decode([i]) for i in ids]
            is_single = len(ids) == 1

            if is_single:
                n_ok += 1
                status = "OK"
            else:
                n_multi += 1
                status = f"MULTI ({len(ids)} tokens: {decoded})"

            results.append({
                "model": model_key, "lang": lang_name,
                "keyword": keyword, "concept": concept,
                "token_ids": ids, "n_tokens": len(ids),
                "single": is_single,
            })

            if not is_single:
                print(f"  WARNING  {keyword:15s} -> {concept:25s} {status}")

        print(f"  Summary: {n_ok}/{n_total} single-token, {n_multi} multi-token")

In [ ]:
# Cell 3 – In-context validation (check keywords don't match inside other words)

CONTEXT_TESTS = {
    "def":    ["def foo():", "default = 1"],
    "for":    ["for i in range(10):", "format(x)"],
    "if":     ["if x > 0:", "iffy = True"],
    "import": ["import os", "important = True"],
    "fn":     ["fn foo() -> i32 { 42 }", "config = 1"],
    "let":    ["let x = 5;", "letter = 'a'"],
    "use":    ["use std::io;", "user = 1"],
    "mod":    ["mod foo {}", "model = 1"],
    "match":  ["match x { _ => () }", "mismatch = 1"],
}

print("\nIN-CONTEXT VALIDATION")
print("=" * 60)

for model_key, config in MODEL_CONFIGS.items():
    tok = AutoTokenizer.from_pretrained(config['id'], trust_remote_code=True)
    print(f"\nModel: {model_key}")

    for keyword, contexts in CONTEXT_TESTS.items():
        target_ids = tok.encode(keyword, add_special_tokens=False)
        if len(target_ids) != 1:
            continue
        target_id = target_ids[0]

        for ctx in contexts:
            ctx_ids = tok.encode(ctx, add_special_tokens=False)
            positions = [i for i, tid in enumerate(ctx_ids) if tid == target_id]
            ctx_tokens = [tok.decode([i]) for i in ctx_ids]
            expected = keyword in ctx.split()[0] or ctx.startswith(keyword)
            found = len(positions) > 0
            status = "OK" if found == expected else "MISMATCH"
            if status == "MISMATCH":
                print(f"  {status} {keyword:10s} in '{ctx}' -> found={found} expected={expected}")
                print(f"    tokens: {ctx_tokens}")

print("\n(Only mismatches shown. No output = all clean.)")

In [ ]:
# Cell 4 – Summary

import pandas as pd

df = pd.DataFrame(results)
print("SUMMARY")
print("=" * 60)

pivot = df.groupby(["model", "lang"])["single"].agg(["sum", "count"]).reset_index()
pivot["pct"] = (pivot["sum"] / pivot["count"] * 100).round(0).astype(int)
pivot.columns = ["Model", "Language", "Single-token", "Total", "Pct"]
display(pivot)

# List all multi-token keywords
multi = df[~df["single"]]
if len(multi) > 0:
    print(f"\nMULTI-TOKEN KEYWORDS ({len(multi)}):")
    for _, r in multi.iterrows():
        print(f"  {r['model']} {r['lang']:8s} {r['keyword']:15s} -> {r['n_tokens']} tokens: {r['token_ids']}")
else:
    print("\nAll keywords are single-token across all models.")

print("\n0 complete.")